In [ ]:
import numpy as np
from numpy.random import rand, randint
import gurobipy as gp
from gurobipy import GRB
from TakeTwoSetup import *

# --- Finding D and H --- #
# D: max number of days
# H: max number of scenes in a day
def find_D(n, S, durations, W):
  D = (2*n*(max(durations)+np.matrix.max(S)))/(W+np.matrix.max(S))
  if np.ceil(D)%2 == 0:
    D = int(np.floor(D))
  else:
    D=int(np.ceil(D))
  return D

D = find_D(n, S, durations, W)

def find_H(n, S, durations, W):
  increasingDurations = np.sort(durations)
  increasingChangeover = np.array(np.sort(np.matrix.flatten(S)))[0]

  vals = []
  hs = []
  for h in range(2, n+1):
    total = np.sum(increasingChangeover[:h]) + np.sum(increasingDurations[:h])
    if total <= W:
      # to make sure combination is within admissible set
      vals.append(total)
      hs.append(h)
  H = hs[np.argmax(vals)]
  return H

H = find_H(n, S, durations, W)

D, H

(9, 4)

In [ ]:
from itertools import permutations

def finding_P(n, S, durations, W, H):
    valid_perms = []
    
    for length in range(1, H + 1):
        for perm in permutations(range(n), length):
            total = 0
            
            for i in range(len(perm) - 1):
                total += S[[perm[i]],[perm[i + 1]]]
            
            for scene in perm:
                total += durations[scene]
            
            if total <= W:
                valid_perms.append(perm)
    
    return valid_perms

# unique subsets from valid permutations
def get_unique_subsets_from_perms(valid_perms):
    subsets = set()
    for perm in valid_perms:
        subsets.add(frozenset(perm))
    return list(subsets)

P_sets = finding_P(n, S, durations, W, H)[1:]
P_sets = get_unique_subsets_from_perms(P_sets)
P = [set(subset) for subset in P_sets]
k = len(P)
k, P

(71,
 [{2, 8},
  {1, 4},
  {4, 6},
  {2, 3},
  {2, 3, 4},
  {3, 6, 8},
  {2, 6},
  {4, 5},
  {0, 5},
  {7, 8},
  {0, 2, 4},
  {1},
  {4, 5, 6},
  {2, 4, 8},
  {1, 4, 6},
  {3, 4},
  {2, 4, 6},
  {0, 4, 6},
  {0, 6},
  {1, 7},
  {0, 3},
  {0, 2, 6},
  {0, 2},
  {0, 3, 6},
  {8},
  {0, 4},
  {2, 4},
  {1, 8},
  {5, 6},
  {3, 4, 6},
  {3, 5},
  {2, 5, 6},
  {3, 4, 8},
  {0, 7},
  {1, 6},
  {5, 8},
  {1, 3},
  {1, 2, 6},
  {4, 7},
  {3, 7},
  {2, 6, 7},
  {0, 1},
  {3, 6},
  {6, 7},
  {2, 3, 6},
  {3, 8},
  {2, 6, 8},
  {1, 5},
  {6, 8},
  {2, 3, 8},
  {4, 6, 8},
  {5},
  {2, 7},
  {2, 4, 7},
  {4},
  {0, 8},
  {5, 7},
  {3, 6, 7},
  {2},
  {4, 6, 7},
  {1, 2},
  {1, 3, 6},
  {5, 6, 8},
  {2, 4, 5},
  {2, 5},
  {3, 5, 6},
  {7},
  {3},
  {6},
  {0, 6, 8},
  {4, 8}])

In [37]:
# --- ILP_pat Model Setup --- #
ILP_pat = gp.Model("ILP_pat", env=env)

# decision variables
a = ILP_pat.addVars(k, n,vtype=GRB.BINARY, name="a")
x_pat = ILP_pat.addVars(k, D, vtype=GRB.BINARY, name="x_pat")
y_pat = ILP_pat.addVars(m, D, D, vtype=GRB.BINARY, name="y_pat")

# objective function
objFunc_pat = ILP_pat.setObjective(
    gp.quicksum(
        holdingCost[i] *
        gp.quicksum(
              (d2-d1+1)*y_pat[i, d1, d2]
              for d1 in range(D)
              for d2 in range(d1, D)
        )
        for i in range(len(actors))
    ), GRB.MINIMIZE
)

# --- constraints (lord help us) --- #
c21 = ILP_pat.addConstr(
    gp.quicksum(
        x_pat[k, 1] for k in range(len(P))
    ) == 1, name="c21"
) # Constraint (21)

c22 = ILP_pat.addConstrs((
    gp.quicksum(
        x_pat[k, d-1]
        for k in range(len(P))
        ) <=
    gp.quicksum(
        x_pat[k, d]
        for k in range(len(P))
        )
    for d in range(2, D)),
    name="c22"
) # Constraint (22)

c23 = ILP_pat.addConstrs((
    gp.quicksum(
        a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for d in range(D)
    ) == 1
    for j in range(n)),
    name="c23"
) # Constraint (23)

c24 = ILP_pat.addConstrs((
    gp.quicksum(
        y_pat[i, d1, d2]
        for d1 in range(D)
        for d2 in range(d1, D)
    ) == 1
    for i in range(m)),
    name="c24"
) # Constraint (24)

c25 = ILP_pat.addConstrs((
    gp.quicksum(
        T[i, j] * a[k, j] * x_pat[k, d]
        for k in range(len(P))
        for j in range(n)
    ) <=
    gp.quicksum(
        y_pat[i, d1, d2]
        for d2 in range(d, D)
        for d1 in range(d)
    )
    for i in range(m) for d in range(D)),
    name="c25"
) # Constraint (25)

# --- Constraints (26-27) --- #
# These constraints are embedded in the init of the variables

In [38]:
# --- optimize that shit --- #
ILP_pat.setParam('TimeLimit', 360)
ILP_pat.optimize()
sol = ILP_pat.getJSONSolution()

Set parameter TimeLimit to value 360
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 7 4700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  360

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 17 rows, 2007 columns and 1470 nonzeros (Min)
Model fingerprint: 0x39d9e840
Model has 405 linear objective coefficients
Model has 90 quadratic constraints
Variable types: 0 continuous, 2007 integer (2007 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [6e+02, 8e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]

Presolve added 21 rows and 0 columns
Presolve removed 0 rows and 382 columns
Presolve time: 0.06s


In [39]:
all_vars = ILP_pat.getVars()
values = ILP_pat.getAttr("X", all_vars)
names = ILP_pat.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

a[1,6] = 1.0
a[10,1] = 1.0
a[11,1] = 1.0
a[16,3] = 1.0
a[18,0] = 1.0
a[26,0] = 1.0
a[26,4] = 1.0
a[31,2] = 1.0
a[33,8] = 1.0
a[39,3] = 1.0
a[48,7] = 1.0
a[51,7] = 1.0
a[53,5] = 1.0
a[55,4] = 1.0
a[58,4] = 1.0
a[60,1] = 1.0
a[64,1] = 1.0
a[64,4] = 1.0
a[65,7] = 1.0
a[68,3] = 1.0
x_pat[0,2] = 1.0
x_pat[0,3] = 1.0
x_pat[1,3] = 1.0
x_pat[2,8] = 1.0
x_pat[3,8] = 1.0
x_pat[4,8] = 1.0
x_pat[5,8] = 1.0
x_pat[6,8] = 1.0
x_pat[7,8] = 1.0
x_pat[8,8] = 1.0
x_pat[9,8] = 1.0
x_pat[10,5] = 1.0
x_pat[12,8] = 1.0
x_pat[13,8] = 1.0
x_pat[14,8] = 1.0
x_pat[15,8] = 1.0
x_pat[17,8] = 1.0
x_pat[18,6] = 1.0
x_pat[19,8] = 1.0
x_pat[20,8] = 1.0
x_pat[21,8] = 1.0
x_pat[22,8] = 1.0
x_pat[23,8] = 1.0
x_pat[24,8] = 1.0
x_pat[25,8] = 1.0
x_pat[27,4] = 1.0
x_pat[27,7] = 1.0
x_pat[27,8] = 1.0
x_pat[28,4] = 1.0
x_pat[28,8] = 1.0
x_pat[29,8] = 1.0
x_pat[30,8] = 1.0
x_pat[31,1] = 1.0
x_pat[32,8] = 1.0
x_pat[33,6] = 1.0
x_pat[34,8] = 1.0
x_pat[35,8] = 1.0
x_pat[36,8] = 1.0
x_pat[37,8] = 1.0
x_pat[38,8] = 1.0
x_pat[39,7] 

In [40]:
with open("sol.json", "w") as f:
  f.write(sol)